In [12]:
import math, random
random.seed(42)
block_size=12 
d_model=24 
n_head=4 
head_dim=d_model//n_head
d_ff=4*d_model
lr=0.03 
steps=400 
ln_eps=1e-5


dataset creation 

In [13]:
vocab_size = 20 
# create token names 
itos={i:f"T{i}" for i in range(vocab_size)}
stoi={v:k for k,v in itos.items()}
transitions={}
for i in range(vocab_size):
       next_tokens=random.sample(range(vocab_size),3)
       transitions[i]=next_tokens
data=[]
current=random.randint(0,vocab_size-1)
for _ in range(5000):
       data.append(current)
       if random.random() < 0.8:
              current=random.choice(transitions[current])
       else:
              current=random.randint(0,vocab_size-1)


In [14]:
print(len(data))

5000


In [15]:
print(data[:20])

[14, 17, 1, 19, 8, 0, 8, 18, 2, 17, 1, 1, 7, 12, 12, 18, 10, 4, 6, 6]


In [16]:
def zeros(r, c):
    return [[0.0 for _ in range(c)] for _ in range(r)]

def randn(r, c, scale=0.02):
    return [[(random.random()*2-1)*scale for _ in range(c)] for _ in range(r)]

def vec_zeros(n):
    return [0.0]*n

def matT(A):
    return list(map(list, zip(*A)))

def matmul(A, B):
    r, m = len(A), len(A[0])
    m2, c = len(B), len(B[0])
    assert m == m2
    out = zeros(r, c)
    for i in range(r):
        for k in range(m):
            for j in range(c):
                out[i][j] += A[i][k] * B[k][j]
    return out

def add(A, B):
    r, c = len(A), len(A[0])
    out = zeros(r, c)
    for i in range(r):
        for j in range(c):
            out[i][j] = A[i][j] + B[i][j]
    return out

def scale_mat(A, s):
    r, c = len(A), len(A[0])
    out = zeros(r, c)
    for i in range(r):
        for j in range(c):
            out[i][j] = A[i][j] * s
    return out

def rowwise_softmax(M):
    out = zeros(len(M), len(M[0]))
    for i, row in enumerate(M):
        mx = max(row)
        exps = [math.exp(x-mx) for x in row]
        s = sum(exps)
        out[i] = [e/s for e in exps]
    return out

In [6]:
def gelu(x):
       return 0.5*x*(1+math.tanh(math.sqrt(2/math.pi)*(x+0.044715*x**3)))
def layernorm_forward(x,gamma,beta):
       T=len(x)
       d=len(x[0])
       Y=zeros(T,d)
       cache=[]
       for t in range(T):
              row=x[t]
              mu=sum(row)/d
              var=sum((v-mu)*(v-mu) for v in row)/d
              inv=1.0/math.sqrt(var+ln_eps)
              xhat=[(v-mu)*inv for v in row]
              Y[t]=[xhat[j]*gamma[j]+beta[j] for j in range(d)]
              Y[t]=Y[t]
              cache.append((row,mu,var,xhat,inv))
       return Y,cache


In [17]:
tok_emb=randn(vocab_size,d_model)
pos_emb=randn(block_size,d_model)
Wq=randn(d_model,d_model)
wk=randn(d_model,d_model)
wv=randn(d_model,d_model)   
wo=randn(d_model,d_model)

w1=randn(d_model,d_ff)
w2=randn(d_ff,d_model)


ln1_gamma=[1.0]*d_model
ln1_beta=[0.0]*d_model
ln2_gamma=[1.0]*d_model
ln2_beta=[0.0]*d_model

wlm=randn(d_model,vocab_size)

In [18]:
def simple_batch():
       i=random.randint(0,len(data)-block_size-1)
       chunk=data[i:i+block_size+1]
       return chunk[:-1],chunk[1:]

forward propagation 

In [20]:
def embed_forward(x_idx):
       T=len(x_idx)
       X=zeros(T,d_model)
       for t in range(T):
              tok=tok_emb[x_idx[t]]
              pos=pos_emb[t]
              X[t]=[tok[j]+pos[j] for j in range(d_model)]
       return X
def linear(X,W):
       return matmul(X,W)
def causal_mask(S):
       T=len(S)
       for i in range(T):
              for j in range(T):
                     if j>i:
                            S[i][j]=-1e9
       return S
# split heads 
def split_heads(M):
       T=len(M)
       heads=[]
       for h in range(n_head):
              H=zeros(T,head_dim)
              start=h*head_dim
              for t in range(T):
                     H[t]=M[t][start:start+head_dim]
              heads.append(H)
       return heads
def concat_heads(heads):
       T=len(heads[0])
       out=zeros(T,d_model)
       for h in range(n_head):
              start=h*head_dim
              for t in range(T):
                     out[t][start:start+head_dim]=heads[h][t]
       return out
def forward(x_idx,y_idx):
       T=len(x_idx)
       X=embed_forward(x_idx)
       X1,_=layernorm_forward(X,ln1_gamma,ln1_beta)
       Q=linear(X1,Wq)
       K=linear(X1,wk)
       V=linear(X1,wv)
       Qh,Kh,Vh=split_heads(Q),split_heads(K),split_heads(V)
       heads_out=[]
       for h in range(n_head):
              scores=matmul(Qh[h],matT(Kh[h]))
              scores=scale_mat(scores,1.0/math.sqrt(head_dim))
              scores=causal_mask(scores)
              P=rowwise_softmax(scores)
              heads_out.append(matmul(P,Vh[h]))
       Att=concat_heads(heads_out)
       O=linear(Att,wo)
       H=add(X,O)

       H1,_=layernorm_forward(H,ln2_gamma,ln2_beta)
       Z1=linear(H1,w1)
       A1=zeros(T,d_ff)
       for t in range(T):
              for j in range(d_ff):
                     A1[t][j]=gelu(Z1[t][j])
       M=linear(A1,w2)
       Yh=add(H,M)
       logits=linear(Yh,wlm)

       loss = 0.0 
       for t in range(T):
              row=logits[t]
              mx=max(row)
              exps=[math.exp(x-mx) for x in row]
              s=sum(exps)
              probs=[e/s for e in exps]
              loss+= math.log(probs[y_idx[t]])
       return loss/T,logits
       

simple training loop 

In [21]:
for step in range(steps):
       x,y=simple_batch()
       loss,_=forward(x,y)
       if step % 50==0:
              print(f"step {step}, loss {loss:.4f}")

TypeError: layernorm_forward() missing 1 required positional argument: 'beta'